In [3]:
!pip install torch torchvision pillow

   ---------------------------------------- 0.0/204.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/204.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/204.1 MB 2.8 MB/s eta 0:01:14
   ---------------------------------------- 1.3/204.1 MB 3.2 MB/s eta 0:01:04
   ---------------------------------------- 2.1/204.1 MB 3.4 MB/s eta 0:01:01
    --------------------------------------- 3.1/204.1 MB 3.8 MB/s eta 0:00:54
    --------------------------------------- 4.2/204.1 MB 4.1 MB/s eta 0:00:50
   - -------------------------------------- 5.2/204.1 MB 4.2 MB/s eta 0:00:47
   - -------------------------------------- 6.6/204.1 MB 4.6 MB/s eta 0:00:44
   - -------------------------------------- 8.1/204.1 MB 4.9 MB/s eta 0:00:40
   - -------------------------------------- 9.7/204.1 MB 5.3 MB/s eta 0:00:38
   -- ------------------------------------- 11.0/204.1 MB 5.4 MB/s eta 0:00:36
   -- ------------------------------------- 12.6/204.1 MB 5.6 MB/s eta 0:00:3

In [13]:
import gradio as gr
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image

# Define CNN model (same as training)
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 32 * 32, 128), nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)
model.load_state_dict(torch.load(r"C:\Users\akash\OneDrive\Documents\breast cancer detection\ad\breast_cancer_cnn.pth", map_location=device))
model.eval()

# Image transformation
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# Prediction function
def predict(img):
    image = img.convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]
    
    labels = ["Benign", "Malignant"]
    confidence_dict = {labels[i]: probs[i] for i in range(2)}
    
    # Find the highest confidence class
    highest_label = labels[int(probs.argmax())]
    
    # Generate paragraph based on highest confidence prediction
    if highest_label == "Benign":
        paragraph = ("The prediction indicates a benign tumor with high confidence. "
                     "Benign tumors are generally non-cancerous and less aggressive. "
                     "They typically do not spread to other parts of the body and are often treatable through surgery.")
    else:
        paragraph = ("The prediction suggests a malignant tumor with high confidence. "
                     "Malignant tumors are cancerous and may grow aggressively. "
                     "Early detection and appropriate treatment are crucial for better management and outcomes.")

    return img, confidence_dict, paragraph

# Gradio interface
interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(type="pil", label="Uploaded Image"),
        gr.Label(num_top_classes=2, label="Prediction Confidence"),
        gr.Textbox(label="Summary About Prediction")
    ],
    title="Breast Cancer Detection By Charu Saini 2129957",
    description="Upload a cancer image to predict confidence scores for Benign and Malignant tumors."
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7868

To create a public link, set `share=True` in `launch()`.
